# Análisis exploratorio: IPM - Variables (incidencias)

Variables del Índice de Pobreza Multidimensional (IPM) a nivel de manzana en Santiago de Cali.

**15 variables** que miden incidencias de privaciones en educación, trabajo, salud, vivienda y servicios.

In [ ]:
# @title 1. Montar Google Drive (opcional, solo en Colab)
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive montado')
else:
    print('Ejecutando localmente')

In [ ]:
# @title 2. Instalar dependencias
!pip install pandas openpyxl matplotlib seaborn geopandas -q

In [ ]:
# @title 3. Clonar repositorio (solo si es necesario)
import os
REPO_URL = 'https://github.com/j0rg3c45/Pobreza_multidimensional_y_condicion_social.git'
REPO_DIR = 'Pobreza_multidimensional_y_condicion_social'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    %cd {REPO_DIR}
    !git pull
    %cd ..
print('Repositorio listo')

In [ ]:
# @title 4. Importar librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams.update({'figure.max_open_warning': 0})
print('Librerías importadas')

In [ ]:
# @title 5. Definir rutas
BASE_DIR = REPO_DIR if os.path.exists(REPO_DIR) else '.'
EXCEL_PATH = os.path.join(BASE_DIR, 'IPM - Variables (incidencias).xlsx')
print(f'Excel: {EXCEL_PATH}')
print(f'Existe: {os.path.exists(EXCEL_PATH)}')

In [ ]:
# @title 6. Diccionario de variables
diccionario = {
    'analf_': 'Analfabetismo',
    'bajo_': 'Bajo logro educativo',
    'infancia_': 'Barreras primera infancia',
    'inasis_': 'Inasistencia escolar',
    'rezago_': 'Rezago escolar',
    'trab_infan_': 'Trabajo infantil',
    'depen_': 'Dependencia económica',
    'infor_': 'Informalidad',
    'salud_': 'Barreras de salud',
    'asegu_': 'Sin aseguramiento en salud',
    'haci_': 'Hacinamiento crítico',
    'pared_': 'Paredes precarias',
    'excre_': 'Eliminación inadecuada de excretas',
    'pisos_': 'Pisos precarios',
    'agua_': 'Sin acceso a agua mejorada'
}
vars_info = pd.DataFrame(diccionario.items(), columns=['Código', 'Descripción'])
display(vars_info)

In [ ]:
# @title 7. Cargar todas las variables en un solo dataset
xls = pd.ExcelFile(EXCEL_PATH)
sheets = [s for s in xls.sheet_names if s != 'Diccionario']

df_ipm_vars = None
for s in sheets:
    df_var = pd.read_excel(xls, s)
    col_name = df_var.columns[1]
    df_var = df_var.rename(columns={col_name: col_name + '_val'})
    df_var.columns = ['cod_mzn', col_name + '_val']
    if df_ipm_vars is None:
        df_ipm_vars = df_var
    else:
        df_ipm_vars = df_ipm_vars.merge(df_var, on='cod_mzn', how='outer')

print(f'Dataset: {df_ipm_vars.shape[0]} manzanas, {df_ipm_vars.shape[1]} columnas')
print(f'Manzanas con datos completos (15 vars): {df_ipm_vars.dropna().shape[0]}')

In [ ]:
# @title 8. Vista previa del dataset
display(df_ipm_vars.head())
print(f'\nTotal manzanas en al menos una variable: {len(df_ipm_vars)}')

In [ ]:
# @title 9. Estadísticas descriptivas generales
desc = df_ipm_vars.describe().T
desc['variable'] = [diccionario.get(c.replace('_val',''), c) for c in desc.index]
desc = desc[['variable', 'count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']]
display(desc.round(2))

In [ ]:
# @title 10. Gráfico de barras - Incidencia promedio por variable
means = df_ipm_vars.drop(columns='cod_mzn').mean().sort_values()
labels = [diccionario.get(c.replace('_val',''), c) for c in means.index]

fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.RdYlGn_r(means / means.max())
bars = ax.barh(labels, means.values, color=colors, edgecolor='gray', linewidth=0.5)

for bar, val in zip(bars, means.values):
    ax.text(val + 0.3, bar.get_y() + bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=9)

ax.set_title('Incidencia promedio de privaciones IPM en Cali', fontsize=14, fontweight='bold')
ax.set_xlabel('Porcentaje de hogares')
ax.set_xlim(0, means.max() + 10)
plt.tight_layout()
plt.show()

In [ ]:
# @title 11. Boxplots de distribución por variable
plot_data = df_ipm_vars.drop(columns='cod_mzn').melt(var_name='var', value_name='valor')
plot_data['var'] = plot_data['var'].map(lambda c: diccionario.get(c.replace('_val',''), c))

fig, ax = plt.subplots(figsize=(16, 7))
order = plot_data.groupby('var')['valor'].median().sort_values().index
sns.boxplot(data=plot_data, x='var', y='valor', order=order, palette='RdYlGn_r', ax=ax)
ax.set_title('Distribución de incidencias IPM por variable', fontsize=14, fontweight='bold')
ax.set_xlabel('')
ax.set_ylabel('Porcentaje de hogares')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# @title 12. Variables con mayor cantidad de manzanas afectadas (>0%)
presencia = {diccionario.get(c.replace('_val',''), c): (df_ipm_vars[c] > 0).sum() for c in df_ipm_vars.columns if c != 'cod_mzn'}
presencia_df = pd.DataFrame(list(presencia.items()), columns=['Variable', 'Manzanas con incidencia'])
presencia_df['% del total'] = (presencia_df['Manzanas con incidencia'] / len(df_ipm_vars) * 100).round(1)
presencia_df = presencia_df.sort_values('Manzanas con incidencia', ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.barh(presencia_df['Variable'], presencia_df['% del total'], color='steelblue', edgecolor='gray')
for bar, val in zip(bars, presencia_df['% del total']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center', fontsize=9)
ax.set_title('% de manzanas con al menos un hogar afectado', fontsize=14, fontweight='bold')
ax.set_xlabel('% de manzanas')
ax.set_xlim(0, 110)
plt.tight_layout()
plt.show()
display(presencia_df)

In [ ]:
# @title 13. Mapa de calor - Correlación entre variables IPM
corr = df_ipm_vars.drop(columns='cod_mzn').corr()
corr.columns = [diccionario.get(c.replace('_val',''), c) for c in corr.columns]
corr.index = [diccionario.get(c.replace('_val',''), c) for c in corr.index]

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, center=0, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlación de Pearson'})
ax.set_title('Correlación entre variables IPM', fontsize=14, fontweight='bold')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# @title 14. Top 10 manzanas con mayor incidencia acumulada
val_cols = [c for c in df_ipm_vars.columns if c != 'cod_mzn']
df_ipm_vars['total_incidencia'] = df_ipm_vars[val_cols].sum(axis=1)
top10 = df_ipm_vars.nlargest(10, 'total_incidencia')[['cod_mzn'] + val_cols + ['total_incidencia']]
top10_display = top10.copy()
top10_display.columns = [diccionario.get(c.replace('_val',''), c) for c in top10_display.columns]
print('Manzanas con mayor incidencia acumulada (suma de % de todas las variables):')
display(top10_display.round(2))

In [ ]:
# @title 15. Dimensiones del IPM - Agrupación por categoría
# Clasificación de variables en dimensiones IPM
dimensiones = {
    'Educación': ['analf_', 'bajo_', 'infancia_', 'inasis_', 'rezago_'],
    'Trabajo': ['trab_infan_', 'depen_', 'infor_'],
    'Salud': ['salud_', 'asegu_'],
    'Vivienda': ['haci_', 'pared_', 'pisos_'],
    'Servicios': ['agua_', 'excre_']
}

dim_data = {}
for dim, vars_list in dimensiones.items():
    valid_vars = [v + '_val' for v in vars_list if v + '_val' in df_ipm_vars.columns]
    if valid_vars:
        dim_data[dim] = df_ipm_vars[valid_vars].mean(axis=1)

df_dim = pd.DataFrame(dim_data)

fig, ax = plt.subplots(figsize=(10, 6))
order_dim = df_dim.mean().sort_values().index
sns.boxplot(data=df_dim[order_dim], palette='Set2', ax=ax)
ax.set_title('Distribución de incidencias por dimensión IPM', fontsize=14, fontweight='bold')
ax.set_ylabel('Incidencia promedio de privaciones (%)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

dim_mean = df_dim.mean().round(2).sort_values()
print('Incidencia promedio por dimensión:')
display(dim_mean.to_frame('Promedio (%)'))

In [ ]:
# @title 16. Variables más críticas - Ranking
print('=== Ranking de variables por incidencia promedio ===')
ranking = means.sort_values(ascending=False).reset_index()
ranking.columns = ['Código', 'Incidencia promedio (%)']
ranking['Variable'] = ranking['Código'].map(lambda c: diccionario.get(c.replace('_val',''), c))
ranking = ranking[['Variable', 'Incidencia promedio (%)']]
ranking['Ranking'] = range(1, len(ranking) + 1)
display(ranking)

print('\nConclusión: La informalidad (>80%) y el bajo logro educativo (>37%) son las privaciones más extendidas en Cali.')
print('Las de menor incidencia promedio son pisos precarios (~6.5%) y trabajo infantil (~4.7%).')

In [ ]:
# @title 17. Análisis por comuna (cargando GeoJSON)
import geopandas as gpd

data_dir = os.path.join(BASE_DIR, 'indice_Pobreza', 'data')
gdf_comunas = gpd.read_file(os.path.join(data_dir, 'geojson_comunas', 'Comunas.geojson'))
print(f'Comunas cargadas: {len(gdf_comunas)}')

# Cargar Mzn_ics para CRS de referencia
import zipfile
ics_dir = os.path.join(data_dir, 'ICS')
if not os.path.exists(ics_dir):
    with zipfile.ZipFile(os.path.join(data_dir, 'ICS.zip'), 'r') as zf:
        zf.extractall(ics_dir)
gdf_mzn = gpd.read_file(os.path.join(ics_dir, 'Mzn_ics.shp'))

# Merge variables con geometrías
df_vars_geom = gdf_mzn[['COD_MZN', 'geometry']].merge(
    df_ipm_vars.rename(columns={'cod_mzn': 'COD_MZN'}), on='COD_MZN', how='inner')
gdf_vars = gpd.GeoDataFrame(df_vars_geom, crs=gdf_mzn.crs)

# Spatial join con comunas
gdf_comunas_proj = gdf_comunas.to_crs(gdf_mzn.crs)
gdf_vars_comuna = gpd.sjoin(gdf_vars, gdf_comunas_proj[['comuna', 'nombre', 'geometry']],
                             how='left', predicate='intersects')

# Promedio por comuna de cada variable
val_cols = [c for c in df_ipm_vars.columns if c != 'cod_mzn']
comuna_stats = gdf_vars_comuna.groupby('nombre')[val_cols].mean().round(2)
comuna_stats.columns = [diccionario.get(c.replace('_val',''), c) for c in comuna_stats.columns]
comuna_stats = comuna_stats.sort_index()
print(f'\nEstadísticas por comuna ({len(comuna_stats)} comunas con datos):')
display(comuna_stats.style.background_gradient(cmap='RdYlGn_r', axis=None))

In [ ]:
# @title 18. Gráfico - Comparación de comunas (top variables)
top3_vars = means.nlargest(3).index.tolist()
top3_labels = [diccionario.get(c.replace('_val',''), c) for c in top3_vars]

comuna_plot = comuna_stats[top3_labels].copy()
comuna_plot = comuna_plot.sort_values(top3_labels[0], ascending=False)

fig, ax = plt.subplots(figsize=(14, 8))
comuna_plot.plot(kind='barh', ax=ax, colormap='RdYlGn_r', edgecolor='gray', linewidth=0.5)
ax.set_title('Top 3 privaciones IPM por comuna', fontsize=14, fontweight='bold')
ax.set_xlabel('Incidencia promedio (%)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# @title 19. Resumen ejecutivo
print('''=== RESUMEN EJECUTIVO IPM - VARIABLES ===

1. FUENTE: IPM - Variables (incidencias).xlsx (15 variables, nivel manzana)
2. COBERTURA: ~15,000 manzanas de Santiago de Cali
3. VARIABLE MÁS CRÍTICA: Informalidad (>82% de hogares)
4. SEGUNDA MÁS CRÍTICA: Bajo logro educativo (>37%)
5. DIMENSIÓN MÁS AFECTADA: Trabajo (Informalidad + Dependencia)
6. DIMENSIÓN MENOS AFECTADA: Vivienda (paredes, pisos)
7. VARIABLES CON MENOS DATOS: Pisos precarios (~1,065 manzanas)
8. CORRELACIONES ALTAS: Bajo logro educativo y Dependencia económica
9. HERRAMIENTA: Notebook listo para Colab o ejecución local
10. PRÓXIMO PASO SUGERIDO: Cruzar con ICS e IPM compuesto por manzana
''')